In [81]:
from langchain.llms import GooglePalm
from dotenv import load_dotenv
load_dotenv()  # This loads the .env file

True

In [82]:
import google.generativeai as genai
import os

# Assuming the API key is already set in the environment from a previous cell
# If not, you would need to set it here, potentially using getpass or userdata.get
api_key = os.environ.get("GOOGLE_API_KEY")
if not api_key:
  # You might want to handle this case more robustly, e.g., raise an error or prompt the user
  print("Google API key not found in environment variables.")
else:
  genai.configure(api_key=api_key)
  # Using google.generativeai directly to generate text
  model = genai.GenerativeModel('gemini-1.5-flash-latest') # Using a valid and supported model name
  response = model.generate_content("Write a 4 line poem of my love for samosa")
  print(response.text)

Golden brown, a crispy shell,
Spiced potato, a magic spell.
Savory delight, a perfect bite,
My love for samosa, shining bright.



In [83]:
from langchain.chains import RetrievalQA


from langchain.embeddings import GooglePalmEmbeddings
from langchain.llms import GooglePalm

In [84]:
from langchain.document_loaders.csv_loader import CSVLoader

# Correct the file path (Colab uses '/content/drive/My Drive/')
file_path = 'data/4c_research_lab_faqs.csv'

# Load CSV file
loader = CSVLoader(file_path=file_path)

# Store the loaded data in the 'data' variable
data = loader.load()

In [85]:
# %pip install --quiet InstructorEmbedding langchain-huggingface faiss-cpu

In [86]:
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize instructor embeddings using the Hugging Face model
instructor_embeddings = HuggingFaceEmbeddings(model_name="hkunlp/instructor-large")

e = instructor_embeddings.embed_query("What does 4C stands for?")

In [87]:
from langchain.document_loaders.csv_loader import CSVLoader

# Use 'question' as the source column since that's what exists in your CSV
loader = CSVLoader(file_path='data/4c_research_lab_faqs.csv', source_column="question")

# Store the loaded data in the 'data' variable
data = loader.load()

# Let's see what we've loaded
print(f"Loaded {len(data)} documents")
print("\nFirst document sample:")
print(data[0])

Loaded 60 documents

First document sample:
page_content='question: What is the 4C Research Lab?
answer: The 4C Research Lab (Cognition, Consciousness, and Critical Care Research Group) is a research laboratory focused on advancing pediatric critical care through innovative research in cognition, consciousness, and critical care. We are based at Western University and London Health Sciences Centre in London, Ontario, Canada.' metadata={'source': 'What is the 4C Research Lab?', 'row': 0}


In [88]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
import pandas as pd

# Load your CSV data
df = pd.read_csv('data/4c_research_lab_faqs.csv')

# Combine question and answer for better context
df['text'] = "Question: " + df['question'] + "\nAnswer: " + df['answer']

# Initialize the embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create a FAISS vector store
from langchain_community.vectorstores import FAISS

# Create documents from your text
documents = df['text'].tolist()
metadatas = [{"source": f"faq_{i}"} for i in range(len(documents))]

# Create the vector store
vectorstore = FAISS.from_texts(
    documents, 
    embeddings,
    metadatas=metadatas
)

# Test a query
query = "What is the research focus of 4C Lab?"
docs = vectorstore.similarity_search(query, k=2)  # Get top 2 most similar documents

print("Query:", query)
print("\nMost relevant FAQs:")
for i, doc in enumerate(docs, 1):
    print(f"\n{i}. {doc.page_content}\n---")

Query: What is the research focus of 4C Lab?

Most relevant FAQs:

1. Question: What is the 4C Research Lab?
Answer: The 4C Research Lab (Cognition, Consciousness, and Critical Care Research Group) is a research laboratory focused on advancing pediatric critical care through innovative research in cognition, consciousness, and critical care. We are based at Western University and London Health Sciences Centre in London, Ontario, Canada.
---

2. Question: Where is the lab located?
Answer: The 4C Research Lab is located at Victoria Hospital & Children's Hospital, 800 Commissioners Rd E., London, ON N6A 5W9, Canada. We are part of London Health Sciences Centre and affiliated with Western University.
---


In [89]:
# First, let's check what's in our vector store
print("Sample documents in the vector store:")
for i, doc in enumerate(documents[:3]):  # Show first 3 documents
    print(f"\nDocument {i+1}:\n{doc}")
    print("-" * 80)

# Create a more lenient retriever
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})  # Get top 3 results without score threshold

# Try different query variations
queries = [
    "how to contact the 4C lab?",
    "ways to reach the research lab",
    "email or phone for the lab",
    "where is the 4C lab located?",
    "how to join the 4C lab?"
]

print("\nSearching for similar questions...")
for query in queries:
    print(f"\nQuery: '{query}'")
    rdocs = retriever.get_relevant_documents(query)
    if rdocs:
        print(f"Found {len(rdocs)} relevant documents:")
        for i, doc in enumerate(rdocs, 1):
            print(f"\nMatch {i}:")
            print(doc.page_content)
    else:
        print("No relevant documents found.")
    print("-" * 80)

Sample documents in the vector store:

Document 1:
Question: What is the 4C Research Lab?
Answer: The 4C Research Lab (Cognition, Consciousness, and Critical Care Research Group) is a research laboratory focused on advancing pediatric critical care through innovative research in cognition, consciousness, and critical care. We are based at Western University and London Health Sciences Centre in London, Ontario, Canada.
--------------------------------------------------------------------------------

Document 2:
Question: What does 4C stand for?
Answer: 4C stands for Cognition, Consciousness, and Critical Care. Our research focuses on these four core areas: Cognition (brain function and cognitive processes), Consciousness (awareness and consciousness states), Critical Care (intensive care medicine), and Comfort (patient comfort and care).
--------------------------------------------------------------------------------

Document 3:
Question: Who is the Principal Investigator?
Answer: Dr. 

In [90]:
len(e)

768

In [91]:
e[:5]

[-0.06358093023300171,
 0.03628506883978844,
 -0.021536385640501976,
 0.00369899976067245,
 0.03212137147784233]

In [92]:
from langchain.vectorstores import FAISS

# Create a FAISS instance for vector database from 'data'
vectordb = FAISS.from_documents(documents=data,
                                 embedding=instructor_embeddings)

# Create a retriever for querying the vector database
retriever = vectordb.as_retriever(score_threshold = 0.7)

In [93]:
rdocs = retriever.get_relevant_documents("how about job placement support?")
rdocs

[Document(id='f9617ebb-31b6-4dd0-827c-2d044a3bd9d5', metadata={'source': 'What types of research positions are available?', 'row': 44}, page_content='question: What types of research positions are available?\nanswer: Available positions include: 1) Research assistants, 2) Graduate students, 3) Postdoctoral fellows, 4) Clinical research coordinators, 5) Data analysts, and 6) Volunteer research positions.'),
 Document(id='4f9634ab-3c25-4d8e-b031-f562fa0bcc01', metadata={'source': "What is the lab's approach to mentorship?", 'row': 56}, page_content="question: What is the lab's approach to mentorship?\nanswer: Our mentorship approach includes: 1) Individual guidance, 2) Skill development, 3) Career planning, 4) Research training, 5) Professional networking, and 6) Continuous support."),
 Document(id='6da5d574-3d30-4035-a3c3-f7cd81cbc296', metadata={'source': 'How can I apply for research positions?', 'row': 43}, page_content='question: How can I apply for research positions?\nanswer: To a

In [96]:
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
# from langchain.llms import GooglePalm # Removed old import
from langchain_google_genai import ChatGoogleGenerativeAI # Added new import
import os

# Assuming the API key is already set in the environment from a previous cell
# If not, you would need to set it here, potentially using getpass or userdata.get
api_key = os.environ.get('GOOGLE_API_KEY')
if not api_key:
  # You might want to handle this case more robustly, e.g., raise an error or prompt the user
  print("Google API key not found in environment variables.")
else:
  # llm = GooglePalm(google_api_key=api_key, temperature=0.7) # Removed old LLM initialization
  llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash-latest", google_api_key=api_key, temperature=0.7) # Added new LLM initialization


prompt_template = """Given the following context and a question, generate an answer based on this context only.
In the answer try to provide as much text as possible from "response" section in the source document context without making much changes.
If the answer is not found in the context, kindly state "I don't know." Don't try to make up an answer.

CONTEXT: {context}

QUESTION: {question}"""


PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)
chain_type_kwargs = {"prompt": PROMPT}


chain = RetrievalQA.from_chain_type(llm=llm,
                            chain_type="stuff",
                            retriever=retriever,
                            input_key="query",
                            return_source_documents=True,
                            chain_type_kwargs=chain_type_kwargs)

In [ ]:
# Use the functions defined in previous cells to perform RAG
query = 'How to contact 4C Lab?'
response = chain({"query": query}) # Use the defined 'chain' object directly
answer = response['result'] # Extract the answer from the response
print(answer)

To discuss potential collaborations, research partnerships, or clinical trials, contact Dr. Rishi Ganesan at rishi.ganesan@lhsc.on.ca.  To join the research team, send your CV to the same email address.


In [100]:
# Use the functions defined in previous cells to perform RAG
query = 'Does 4c Lab have a website?'
response = chain({"query": query}) # Use the defined 'chain' object directly
answer = response['result'] # Extract the answer from the response
print(answer)

I don't know.
